# Yomu real-manga clean reconstruction — guarded Colab run

This notebook is prepared for the owner-provided 795×960 page. It does not auto-connect/disconnect a runtime, install packages, or run a model until the owner confirms the current resource and research-rights gates. It can mount the owner's private Google Drive after normal Colab authorization, read a SHA-256-pinned input ZIP, and save results to a new non-overwriting Drive run folder. The input ZIP contains the source and two independently reviewed mask scenarios. Main scenario preserves BAM/KYAA; full experimental adds SFX and ambiguous microtext.

Only Big-LaMa has a pinned runnable adapter here. The two other candidates remain NOT_RUN until their exact weights, licenses, safe adapters, and T4 compatibility are cleared. Do not treat this one-model notebook as a three-model benchmark or a winner.


In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, stat, time, urllib.request, zipfile
import numpy as np
from PIL import Image, ImageDraw
import torch
from google.colab import drive
print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'cuda_available': torch.cuda.is_available()})
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    print({'gpu': torch.cuda.get_device_name(0), 'vram_total_bytes': int(total_bytes), 'vram_free_bytes': int(free_bytes), 'ram_total_bytes': int(__import__('psutil').virtual_memory().total)})
else:
    print('NO_GPU: stop; no model weights will be loaded')


In [ ]:
# Private Drive sync: the ZIP is uploaded by the connected Drive tool; no public URL or browser upload.
DRIVE_ROOT = Path('/content/drive')
if not (DRIVE_ROOT / 'MyDrive').is_dir(): drive.mount(str(DRIVE_ROOT))
DRIVE_RESEARCH_ROOT = DRIVE_ROOT / 'MyDrive' / 'Yomu Colab Research'
DRIVE_INPUT = DRIVE_RESEARCH_ROOT / 'real_manga_colab_inputs.zip'
EXPECTED_ZIP_SHA256 = '725c75c2846e2e73eadd3a2d2f661b3e59ff8db996bd849d712dd0a710ddf427'
ZIP_PATH = Path('/content/real_manga_colab_inputs.zip')
def sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
if ZIP_PATH.exists():
    assert sha256(ZIP_PATH) == EXPECTED_ZIP_SHA256, 'Existing /content ZIP differs; refusing overwrite'
else:
    assert DRIVE_INPUT.is_file(), f'Private Drive input missing: {DRIVE_INPUT}'
    assert sha256(DRIVE_INPUT) == EXPECTED_ZIP_SHA256, 'Drive ZIP hash mismatch'
    shutil.copy2(DRIVE_INPUT, ZIP_PATH)
assert sha256(ZIP_PATH) == EXPECTED_ZIP_SHA256, 'Input ZIP SHA-256 mismatch'
OUT = Path('/content/yomu_real_manga_clean_v1')
assert not OUT.exists(), 'Refuse to overwrite prior evidence'
EXTRACT = Path('/content/yomu_real_manga_input_v1')
if EXTRACT.exists():
    assert (EXTRACT / 'source.png').is_file() and sha256(EXTRACT / 'source.png') == 'da5714748b63216fba3980ec6bf224d4223c56b8d51f00a595debbe8c6510b24', 'Existing extracted input is incomplete or different'
else:
    with zipfile.ZipFile(ZIP_PATH) as z:
        total = 0
        for info in z.infolist():
            parts = Path(info.filename).parts
            assert not Path(info.filename).is_absolute() and '..' not in parts and '\\' not in info.filename and ':' not in info.filename, 'Unsafe ZIP path'
            assert not stat.S_ISLNK(info.external_attr >> 16), 'Symlink not allowed'
            total += info.file_size
        assert total < 15 * 1024 * 1024, 'Unexpected input archive size'
        assert z.testzip() is None, 'ZIP integrity failed'
        z.extractall(EXTRACT)
manifest = json.loads((EXTRACT / 'mask_report.json').read_text())
source_path = EXTRACT / 'source.png'
assert sha256(source_path) == manifest['source_sha256'], 'Source hash mismatch'
source = Image.open(source_path).convert('RGB')
assert source.size == (795, 960), f'Unexpected source dimensions: {source.size}'
speech_mask = Image.open(EXTRACT / 'mask_speech.png').convert('L')
full_mask = Image.open(EXTRACT / 'mask_full_experimental.png').convert('L')
sfx_mask = Image.open(EXTRACT / 'mask_sfx.png').convert('L')
assert speech_mask.size == full_mask.size == sfx_mask.size == source.size
assert not (np.asarray(speech_mask) > 0).any() or not ((np.asarray(speech_mask)>0) & (np.asarray(sfx_mask)>0)).any(), 'Scenario A overlaps SFX mask'
display(Image.open(EXTRACT / 'mask_scenarios_preview.png'))
print({'source_sha256': sha256(source_path), 'dimensions': source.size, 'speech_mask_pixels': int((np.asarray(speech_mask)>0).sum()), 'full_mask_pixels': int((np.asarray(full_mask)>0).sum()), 'mask_status': manifest['status']})
print('STOP HERE and visually approve the mask preview before any inference. The source is cropped at the bottom; unresolved/ambiguous text stays review-only.')


## Explicit one-inference gate
The exact Anime-Manga Big-LaMa checkpoint is 205,717,870 bytes, SHA-256 `479d3afdcb7ed2fd944ed4ebcc39ca45b33491f0f2e43eb1000bd623cfb41823`. Sanster's release points to `dreMaz/AnimeMangaInpainting`, whose model card declares MIT; the exact training-data provenance is not independently documented here. This notebook is research evaluation only, never product integration. Before changing the four booleans below, confirm the connected runtime's GPU/resource authorization and acknowledge the provenance limit. This cell runs only scenario A and only one model inference. It downloads the checkpoint only if absent. No automatic retry.

Model 2/3 are NOT_RUN: current research records do not clear a safe complete stack for this image/runtime. MAT is research-only/legacy; SDXL's pinned snapshot lacks its safety-checker component; BrushNet/PowerPaint require exact base/checkpoint/license and safe-adapter review. Do not bypass safety checks.

In [ ]:
OWNER_CONFIRMS_NO_ADDITIONAL_PAID_RESOURCE = False
OWNER_ACKNOWLEDGES_RESEARCH_ONLY_AND_TRAINING_DATA_PROVENANCE_LIMITS = False
OWNER_APPROVES_ONE_CHECKPOINT_DOWNLOAD_IF_NEEDED = False
RUN_ONE_BIG_LAMA_SCENARIO_A = False
assert all([OWNER_CONFIRMS_NO_ADDITIONAL_PAID_RESOURCE, OWNER_ACKNOWLEDGES_RESEARCH_ONLY_AND_TRAINING_DATA_PROVENANCE_LIMITS, OWNER_APPROVES_ONE_CHECKPOINT_DOWNLOAD_IF_NEEDED, RUN_ONE_BIG_LAMA_SCENARIO_A]), 'Gate closed: owner must verify current runtime, acknowledge research-only/data-provenance limits, approve one download if needed, and approve one Scenario A inference'
assert torch.cuda.is_available(), 'CUDA unavailable; no weights loaded'
gpu = torch.cuda.get_device_name(0)
print({'selected_gpu': gpu, 'total_vram_bytes': int(torch.cuda.get_device_properties(0).total_memory), 'free_vram_bytes': int(torch.cuda.mem_get_info(0)[0])})


In [ ]:
# One Big-LaMa inference, scenario A only. Strict source-pixel composition outside mask.
assert not OUT.exists(), 'Prior output exists; refusing duplicate inference or overwrite'
result_zip=Path('/content/yomu_real_manga_big_lama_result.zip')
assert not result_zip.exists(), 'Prior result bundle exists; refusing duplicate inference'
RUN_ID = time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()) + '-' + os.urandom(4).hex()
RUN_LOCK = DRIVE_RESEARCH_ROOT / 'run_locks' / 'big_lama_scenario_a_once.json'
assert not RUN_LOCK.exists(), 'One-shot Scenario A lock exists; inspect prior attempt and evidence; no retry'
import cv2
WEIGHT_URL = 'https://github.com/Sanster/models/releases/download/AnimeMangaInpainting/anime-manga-big-lama.pt'
WEIGHT_SHA256 = '479d3afdcb7ed2fd944ed4ebcc39ca45b33491f0f2e43eb1000bd623cfb41823'
WEIGHT_SIZE = 205717870
WEIGHT = Path('/content/models/anime-manga-big-lama.pt'); WEIGHT.parent.mkdir(parents=True, exist_ok=True)
def checked_download():
    if not WEIGHT.exists():
        part = WEIGHT.with_suffix('.part')
        assert not part.exists(), 'Partial prior download exists; inspect manually; no retry'
        urllib.request.urlretrieve(WEIGHT_URL, part)
        assert part.stat().st_size == WEIGHT_SIZE and sha256(part) == WEIGHT_SHA256, 'Checkpoint integrity failed; refusing load'
        part.replace(WEIGHT)
    assert WEIGHT.stat().st_size == WEIGHT_SIZE and sha256(WEIGHT) == WEIGHT_SHA256, 'Checkpoint integrity failed; refusing load'
checked_download()
mask = np.asarray(speech_mask, dtype=np.uint8)
src = np.asarray(source, dtype=np.uint8)
ys, xs = np.where(mask > 0); assert len(xs), 'Empty authorized mask'
margin = max(48, int(max(xs.max()-xs.min()+1, ys.max()-ys.min()+1)*0.20))
x0=max(0,int(xs.min())-margin); y0=max(0,int(ys.min())-margin); x1=min(src.shape[1],int(xs.max())+1+margin); y1=min(src.shape[0],int(ys.max())+1+margin)
crop=src[y0:y1,x0:x1]; cmask=mask[y0:y1,x0:x1]
ph=(8-crop.shape[0]%8)%8; pw=(8-crop.shape[1]%8)%8
crop=cv2.copyMakeBorder(crop,0,ph,0,pw,cv2.BORDER_REFLECT) if ph or pw else crop
cmask=cv2.copyMakeBorder(cmask,0,ph,0,pw,cv2.BORDER_REFLECT) if ph or pw else cmask
def tensor_img(a): return torch.from_numpy(a.transpose(2,0,1).astype(np.float32)/255.0).unsqueeze(0).to('cuda:0')
image_tensor=tensor_img(crop); mask_tensor=torch.from_numpy((cmask>0).astype(np.float32)).unsqueeze(0).unsqueeze(0).to('cuda:0')
RUN_LOCK.parent.mkdir(parents=True, exist_ok=True)
with RUN_LOCK.open('x', encoding='utf-8') as lock_file: json.dump({'run_id': RUN_ID, 'state': 'INFERENCE_STARTED', 'source_sha256': sha256(source_path), 'mask': 'scenario_a_speech_caption_preserve_sfx'}, lock_file)
torch.cuda.reset_peak_memory_stats(); t0=time.perf_counter(); model=torch.jit.load(str(WEIGHT),map_location='cuda:0').eval(); torch.cuda.synchronize(); load_s=time.perf_counter()-t0
t1=time.perf_counter()
with torch.inference_mode(): pred=model(image_tensor,mask_tensor)
torch.cuda.synchronize(); infer_s=time.perf_counter()-t1
donor=np.clip(pred[0].permute(1,2,0).detach().cpu().numpy()*255,0,255).astype(np.uint8)[:y1-y0,:x1-x0]
raw=src.copy(); raw[y0:y1,x0:x1]=donor
final=src.copy(); roi=final[y0:y1,x0:x1]; roi[cmask[:y1-y0,:x1-x0]>0]=donor[cmask[:y1-y0,:x1-x0]>0]; final[y0:y1,x0:x1]=roi
outside=int(np.any(final!=src,axis=2)[mask==0].sum()); assert outside==0, f'Outside-mask changes: {outside}'
OUT.mkdir(parents=True,exist_ok=False)
Image.fromarray(src).save(OUT/'original.png'); speech_mask.save(OUT/'mask_speech.png'); Image.fromarray(raw).save(OUT/'big_lama_raw.png'); Image.fromarray(final).save(OUT/'big_lama_clean.png')
preview=Image.new('RGB',(source.width*3,source.height+30),'white')
for i,(label,img) in enumerate([('original',source),('mask',speech_mask.convert('RGB')),('Big-LaMa strict composite',Image.fromarray(final))]):
    preview.paste(img,(i*source.width,30)); ImageDraw.Draw(preview).text((i*source.width+5,5),label,fill='black')
preview.save(OUT/'comparison.png')
report={'status':'REVIEW_REQUIRED','model':'Anime-Manga Big-LaMa TorchScript','gpu':gpu,'source_sha256':sha256(source_path),'checkpoint_sha256':sha256(WEIGHT),'resolution':list(source.size),'scenario':'A_speech_and_caption_preserve_SFX','model_load_seconds':load_s,'inference_seconds':infer_s,'peak_vram_allocated_bytes':int(torch.cuda.max_memory_allocated()),'peak_vram_reserved_bytes':int(torch.cuda.max_memory_reserved()),'outside_mask_changed_pixels':outside,'text_removal_quality':'HUMAN_REVIEW_REQUIRED','mask_report_status':manifest['status'],'model_2':'NOT_RUN_LICENSE_SAFE_ADAPTER_GATE','model_3':'NOT_RUN_LICENSE_SAFE_ADAPTER_GATE','artifacts':{p.name:sha256(p) for p in OUT.iterdir() if p.is_file()}}
(OUT/'report.json').write_text(json.dumps(report,indent=2))
with zipfile.ZipFile(result_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file(): z.write(p,arcname=p.name)
# Persist into a unique private Drive folder; never overwrite previous run artifacts.
RESULTS_ROOT = DRIVE_RESEARCH_ROOT / 'results'; RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUN = RESULTS_ROOT / RUN_ID; DRIVE_RUN.mkdir(exist_ok=False)
for artifact in sorted(OUT.iterdir()):
    if artifact.is_file(): shutil.copy2(artifact, DRIVE_RUN / artifact.name)
shutil.copy2(result_zip, DRIVE_RUN / result_zip.name)
artifact_manifest = {'run_id': RUN_ID, 'input_zip_sha256': EXPECTED_ZIP_SHA256, 'source_sha256': sha256(source_path), 'model': report['model'], 'checkpoint_sha256': report['checkpoint_sha256'], 'artifacts': {p.name: {'size': p.stat().st_size, 'sha256': sha256(p)} for p in sorted(DRIVE_RUN.iterdir()) if p.is_file()}}
(DRIVE_RUN / 'artifact_manifest.json').write_text(json.dumps(artifact_manifest, indent=2))
display(preview); print(json.dumps(report,indent=2)); print({'drive_result_folder': str(DRIVE_RUN), 'run_id': RUN_ID, 'artifact_manifest': artifact_manifest})
print('Do not accept automatically: inspect text residuals, speech-bubble edges, line art, reticules and preserved BAM/KYAA. No text-removal PASS is inferred from pixel equality.')